# Lesson 18 - Useful Standard Library Patterns

**Goal:** Use Counter, defaultdict, regular expressions, and context managers for practical record processing.

**Level:** intermediate. Complete the preceding chapters first. This notebook defines its own
inputs and runs independently. Run cells top to bottom with **Shift+Enter**. Predict results,
inspect the output, and change one input at a time. No external services or third-party packages
are required beyond your Jupyter setup.

Examples include lightweight executable checks. Practice cells intentionally contain no solutions;
expected results help you verify your own work. Temporary files and databases are isolated and cleaned up.

Where salary examples appear, use the course's simplified exercise rules: DA 48% of basic,
HRA 16% capped at 16,000 monthly, PF 12% of basic plus DA, and annual tax 20% only on annual
gross above 12,00,000, divided by 12 for monthly tax. PF is a separate deduction.
These are learning assumptions, not actual tax or payroll rules.

## 1. Count normalized categories

Counter counts occurrences. Normalize text before counting if case and extra whitespace should not create separate categories. most_common orders by frequency; tied counts retain first-seen order.

In [1]:
from collections import Counter, defaultdict

departments = [" Finance ", "HR", "finance", "IT", "hr"]
counts = Counter(value.strip().casefold() for value in departments)
print(counts.most_common())
assert counts == {"finance": 2, "hr": 2, "it": 1}

[('finance', 2), ('hr', 2), ('it', 1)]


## 2. Group without repeated setup

defaultdict creates a default value when a missing key is accessed with brackets. Use list to accumulate groups or int for numeric counters. get(...) does not trigger default creation.

In [2]:
employees = [{"name": "Asha", "dept": "Finance"},
             {"name": "Arun", "dept": "IT"},
             {"name": "Neha", "dept": "Finance"}]
by_department = defaultdict(list)
for employee in employees:
    by_department[employee["dept"]].append(employee["name"])
print(dict(by_department))
assert by_department["Finance"] == ["Asha", "Neha"]

{'Finance': ['Asha', 'Neha'], 'IT': ['Arun']}


## 3. Validate a small text pattern

Use ordinary string methods for simple cleanup. A regular expression is useful for a specific structural pattern. fullmatch checks the whole string; [0-9] requires ASCII digits. This is an employee-code format check, not general validation of identity.

In [3]:
import re

employee_code = re.compile(r"EMP-([0-9]{4})")
for value in ["EMP-1042", "EMP-12", "xEMP-1042", "EMP-1042x"]:
    match = employee_code.fullmatch(value)
    print(value, bool(match))
    if match:
        print("Numeric ID:", int(match.group(1)))
assert employee_code.fullmatch("EMP-1042") is not None

EMP-1042 True
Numeric ID: 1042
EMP-12 False
xEMP-1042 False
EMP-1042x False


## 4. A small context manager

A context manager pairs setup and cleanup. contextlib.contextmanager turns a generator with one yield into a context manager. Put cleanup in finally so it runs after success or failure. Prefer existing managers for files and temporary directories.

In [4]:
from contextlib import contextmanager

@contextmanager
def tracked_operation(events):
    events.append("start")
    try:
        yield
    finally:
        events.append("finish")

events = []
try:
    with tracked_operation(events):
        events.append("work")
        raise ValueError("Sample failure")
except ValueError:
    pass
print(events)
assert events == ["start", "work", "finish"]

['start', 'work', 'finish']


## Practice lab

Define all inputs in each answer cell, use `practice_` names where helpful, and explain important
decisions in comments. Try a task before looking at its expected results. Add at least one normal,
one boundary, and one invalid-input check where the task has an input contract.
Keep file exercises inside TemporaryDirectory until you deliberately choose a permanent output path.

### Exercise 1. Count departments

Normalize and count ["HR", " hr ", "IT", "Finance", "IT"].

**Check:** hr=2, it=2, finance=1.

In [5]:
# Write your solution and checks here.

### Exercise 2. Group salaries

Group employee basic salaries by department and calculate totals.

**Check:** Finance salaries 40,000 and 60,000 total 100,000.

In [6]:
# Write your solution and checks here.

### Exercise 3. Validate codes

Accept only EMP- followed by exactly four ASCII digits.

**Check:** EMP-0001 accepted; EMP-1 and EMP-12345 rejected.

In [7]:
# Write your solution and checks here.

### Exercise 4. Choose simple tools

Validate a .csv suffix case-insensitively using string methods, not regex.

**Check:** REPORT.CSV accepted; report.csv.txt rejected.

In [8]:
# Write your solution and checks here.

### Exercise 5. Cleanup on failure

Use tracked_operation with a successful and failing block.

**Check:** Both event lists end with finish.

In [9]:
# Write your solution and checks here.

## Chapter project

Group a batch of validated employee records by normalized department, summarize counts and salary totals, and report invalid employee codes separately.

In [10]:
# Build your chapter project here. Separate inputs, logic, and display.

## Reflection

1. Which input assumptions does your solution enforce, and which does it document?
2. What happens for empty input, a boundary value, or a malformed record?
3. Which parts can be reused without the notebook?
4. What would make the code clearer to another learner?

Restart the kernel and run all cells after completing your work to catch hidden dependencies.

**Next:** `19_sqlite_and_persistence.ipynb` - SQLite and Persistent Records.